In [4]:
import os
from urllib.parse import quote_plus
from sqlalchemy import create_engine
import pandas as pd
import matplotlib.pyplot as plt
import re
from dotenv import load_dotenv

# Load .env variables
load_dotenv()

db_user = quote_plus(os.getenv("DB_USER"))
db_password = quote_plus(os.getenv("DB_PASSWORD"))
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

# Create SQLAlchemy engine
engine = create_engine(
    f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

# Test connection
try:
    conn = engine.connect()
    print("✅ Connected successfully")
except Exception as e:
    print("❌ Connection failed:", e)


✅ Connected successfully


In [5]:
# Fetch all tables in 'public' schema
tables = pd.read_sql(
    "SELECT table_name FROM information_schema.tables WHERE table_schema='public'", 
    conn
)
tables_list = tables['table_name'].tolist()
print("Tables in DB:", tables_list)

# Fetch columns for each table
for table in tables_list:
    cols = pd.read_sql(
        f"SELECT column_name FROM information_schema.columns WHERE table_name='{table}'",
        conn
    )
    print(f"\nColumns in table '{table}':\n", cols['column_name'].tolist())


Tables in DB: ['argo_meta_meta', 'argo_meta_variables', 'argo_prof_meta', 'argo_prof_variables', 'argo_rtraj_meta', 'argo_rtraj_variables', 'argo_tech_meta', 'argo_tech_variables']

Columns in table 'argo_meta_meta':
 ['N_TRANS_SYSTEM', 'N_POSITIONING_SYSTEM', 'N_LAUNCH_CONFIG_PARAM', 'N_CONFIG_PARAM', 'N_MISSIONS', 'N_SENSOR', 'N_PARAM', 'title', 'institution', 'source', 'history', 'references', 'user_manual_version', 'Conventions']

Columns in table 'argo_meta_variables':
 ['name', 'dtype', 'shape', 'attributes', 'sample']

Columns in table 'argo_prof_meta':
 ['N_PROF', 'N_PARAM', 'N_LEVELS', 'N_CALIB', 'N_HISTORY', 'title', 'institution', 'source', 'history', 'references', 'user_manual_version', 'Conventions', 'featureType']

Columns in table 'argo_prof_variables':
 ['name', 'dtype', 'shape', 'attributes', 'sample']

Columns in table 'argo_rtraj_meta':
 ['N_PARAM', 'N_MEASUREMENT', 'N_CYCLE', 'N_HISTORY', 'title', 'institution', 'source', 'history', 'references', 'user_manual_versio

In [6]:
def ask_argo(query):
    """
    Query Argo DB tables for temperature/salinity profiles by year, and plot depth profiles.
    """
    # 1️⃣ Extract year from query
    year_match = re.search(r"(19|20)\d{2}", query)
    year = year_match.group(0) if year_match else None

    # 2️⃣ Detect parameter
    parameter = None
    if "temperature" in query.lower():
        parameter = "temperature"
    elif "salinity" in query.lower():
        parameter = "salinity"

    # 3️⃣ Decide table and columns
    # Using argo_prof_meta + argo_prof_variables as main example
    table = "argo_prof_meta"  # you can add logic to choose table dynamically
    if parameter == "temperature":
        fetch_cols = ["N_PROF", "temperature", "depth_index"]
    elif parameter == "salinity":
        fetch_cols = ["N_PROF", "salinity", "depth_index"]
    else:
        fetch_cols = ["N_PROF", "temperature", "salinity", "depth_index"]

    # 4️⃣ Construct SQL query (adjust column names based on your DB)
    sql_query = f"""
        SELECT {', '.join(fetch_cols)}
        FROM {table}
        WHERE EXTRACT(YEAR FROM time) = {year}  -- assumes 'time' column exists
    """

    # 5️⃣ Fetch Data
    try:
        df = pd.read_sql(sql_query, conn)
    except Exception as e:
        print("❌ SQL Error:", e)
        return

    if df.empty:
        print(f"⚠️ No data found for {parameter} in {year}")
        return

    print("✅ Data fetched. Preview:")
    print(df.head())

    # 6️⃣ Plot profiles (first 5)
    plt.figure(figsize=(6, 8))
    for profile in df["N_PROF"].unique()[:5]:
        profile_data = df[df["N_PROF"] == profile]
        if "temperature" in df.columns:
            plt.plot(profile_data["temperature"], profile_data["depth_index"], label=f"Temp - Profile {profile}")
        if "salinity" in df.columns:
            plt.plot(profile_data["salinity"], profile_data["depth_index"], label=f"Salinity - Profile {profile}")

    plt.gca().invert_yaxis()
    plt.xlabel("Value")
    plt.ylabel("Depth Index")
    plt.title(f"Argo Profiles for {year}")
    plt.legend()
    plt.show()


In [7]:
user_query = input("Enter your query about Argo data: ")
ask_argo(user_query)


❌ SQL Error: (psycopg2.errors.UndefinedColumn) column "n_prof" does not exist
LINE 2:         SELECT N_PROF, salinity, depth_index
                       ^

[SQL: 
        SELECT N_PROF, salinity, depth_index
        FROM argo_prof_meta
        WHERE EXTRACT(YEAR FROM time) = 2005  -- assumes 'time' column exists
    ]
(Background on this error at: https://sqlalche.me/e/20/f405)
